# 05 · Ensinando a máquina ao vivo — **EPI na obra**

O bloco do treino. Aqui a plateia vê a IA **aprendendo na frente dela**: a curva
de erro descendo e a de acerto subindo, época por época.

Roteiro de palco:
1. mostrar que o modelo pronto **não sabe** o que é capacete → só vê "pessoa"
2. mostrar as duas pastas de fotos → *"é só isso que eu vou dar para ele"*
3. **treinar ao vivo**, 20–30 épocas, e narrar a curva enquanto sobe
4. testar em fotos que ele **nunca viu**
5. a matriz de confusão: **onde** ele ainda erra

> Fala de palco para o começo: *"eu não vou escrever nenhuma regra sobre
> capacete. Nenhum 'se for amarelo e redondo'. Eu só vou mostrar exemplos —
> e é exatamente assim que a coisa toda funciona, do ChatGPT a isso aqui."*

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

# ── 2. a raiz de tudo, e a conferência de que ela é REAL ──────────────
#
# ARMADILHA que já custou uma sessão: a linha acima cria a variável
# `drive` (minúscula), que é o MÓDULO do Colab. Se algum caminho for
# escrito com `drive` em vez de `DRIVE`, o Python aceita numa boa e
# monta um caminho como
#     <module 'google.colab.drive' from '/usr/local/...'>/04-garrafas
# O código roda, cria pastas, exporta arquivos — tudo no disco
# temporário do Colab, que evapora quando a sessão encerra. Nada disso
# chega ao seu Drive, e não há erro nenhum na tela.
#
# A conferência abaixo transforma esse silêncio num aviso imediato.

DRIVE = "/content/drive/MyDrive/PALESTRA-IA"

if not DRIVE.startswith("/content/drive/"):
    raise SystemExit(
        "DRIVE aponta para fora do Google Drive: " + repr(DRIVE) + "\n"
        "Provavelmente algum caminho usou `drive` (o módulo) em vez de `DRIVE`.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("O Drive não montou. Rode esta célula de novo e autorize o acesso.")

os.makedirs(DRIVE, exist_ok=True)
print("raiz no Drive:", DRIVE)
print("existe de verdade:", os.path.isdir(DRIVE))

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
# DRIVE não é redefinido aqui de propósito: quem define é a célula de
# setup, e uma variável de caminho com duas origens é como se perde a
# noção de onde os arquivos foram parar.
print("palco configurado")

### Passo 1 · o modelo pronto não sabe o que você precisa

In [ ]:
import glob, cv2, matplotlib.pyplot as plt
generico = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")

amostras = [f for f in sorted(glob.glob(f"{DRIVE}/05-epi/treino/val/com_epi/*"))
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
if not amostras:
    raise SystemExit("faltam fotos em 05-epi/treino/val/com_epi/ (rode o notebook 00)")

r = generico.predict(amostras[0], conf=.3, verbose=False)[0]
achou = sorted({generico.names[int(c)] for c in r.boxes.cls})
print("o que o modelo pronto enxerga:", achou)
print("existe alguma classe de EPI?  ", any("helmet" in a or "hat" in a for a in achou))

plt.figure(); plt.imshow(cv2.cvtColor(r.plot(line_width=4), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title("Modelo genérico: vê a pessoa, ignora o capacete")
plt.tight_layout(); plt.show()

### Passo 2 · o material de estudo — só duas pastas

In [ ]:
import glob, os, matplotlib.pyplot as plt, cv2
def n(p):
    return len([f for f in glob.glob(f"{DRIVE}/05-epi/treino/{p}/*")
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))])

print(f"treino  com_epi: {n('train/com_epi'):>4}   sem_epi: {n('train/sem_epi'):>4}")
print(f"prova   com_epi: {n('val/com_epi'):>4}   sem_epi: {n('val/sem_epi'):>4}")

# mostra uma amostra de cada lado, lado a lado
fig, axs = plt.subplots(2, 4, figsize=(22, 11))
for linha, classe in enumerate(["com_epi", "sem_epi"]):
    arqs = [f for f in sorted(glob.glob(f"{DRIVE}/05-epi/treino/train/{classe}/*"))
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))][:4]
    for col in range(4):
        ax = axs[linha][col]; ax.axis("off")
        if col < len(arqs):
            ax.imshow(cv2.cvtColor(cv2.imread(arqs[col]), cv2.COLOR_BGR2RGB))
        if col == 0:
            ax.set_title(classe.upper(), fontsize=30, loc="left",
                         color="#3fe0a8" if classe == "com_epi" else "#ff5c5c")
plt.tight_layout(); plt.show()

### Passo 3 · o treino ao vivo

**Ajuste `EPOCAS` na frente da plateia.** Rodar primeiro com 5 e depois com 25 é
a melhor demonstração possível de "por que treinar mais importa" — e o contraste
entre as duas curvas dispensa qualquer explicação técnica.

In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

def treinar_mostrando(modelo, dados, epocas, imgsz=224, batch=32,
                      projeto="/content/runs", nome="ao_vivo", titulo="Aprendendo"):
    """Treina e redesenha a curva a cada epoca — o ponto alto da demo."""
    hist = {}                                  # epoca -> (perda, acuracia)

    def a_cada_epoca(trainer):
        m = getattr(trainer, "metrics", None) or {}
        acc = m.get("metrics/accuracy_top1")
        perda = float(trainer.loss.item()) if getattr(trainer, "loss", None) is not None else None
        hist[trainer.epoch + 1] = (perda, acc)   # dict: a epoca repetida sobrescreve

        eps = sorted(hist)
        perdas = [hist[e][0] for e in eps]
        accs = [(hist[e][1] or 0) * 100 for e in eps]

        clear_output(wait=True)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(20, 8))
        a1.plot(eps, perdas, lw=5, color="#ff5c5c", marker="o", ms=10)
        a1.set_title("ERRO — tem que descer"); a1.set_xlabel("época")
        a2.plot(eps, accs, lw=5, color="#3fe0a8", marker="o", ms=10)
        a2.set_ylim(0, 101)
        a2.set_title("ACERTO — tem que subir"); a2.set_xlabel("época"); a2.set_ylabel("%")
        if accs:
            a2.text(eps[-1], accs[-1], f"  {accs[-1]:.0f}%", fontsize=34,
                    color="#3fe0a8", va="center", fontweight="bold")
        fig.suptitle(f"{titulo} · época {max(eps)} de {epocas}", fontsize=34)
        plt.tight_layout(); plt.show()

    modelo.add_callback("on_fit_epoch_end", a_cada_epoca)
    r = modelo.train(data=dados, epochs=epocas, imgsz=imgsz, batch=batch,
                     project=projeto, name=nome, exist_ok=True, verbose=False, plots=True)
    print("pesos e graficos em:", r.save_dir)
    return r

In [ ]:
EPOCAS = 25          # ← o botão do palco

modelo = YOLO(f"{DRIVE}/00-pesos/yolo11n-cls.pt")
res = treinar_mostrando(
    modelo,
    dados=f"{DRIVE}/05-epi/treino",
    epocas=EPOCAS,
    titulo="Aprendendo a reconhecer EPI",
    nome="epi",
)

In [ ]:
import shutil
shutil.copy(f"{res.save_dir}/weights/best.pt", f"{DRIVE}/05-epi/pesos/epi_best.pt")
print("modelo especialista salvo no Drive")

### Passo 4 · a prova — fotos que ele nunca viu

In [ ]:
import glob, random, cv2, matplotlib.pyplot as plt
especialista = YOLO(f"{DRIVE}/05-epi/pesos/epi_best.pt")

prova = []
for classe in ["com_epi", "sem_epi"]:
    prova += [(f, classe) for f in glob.glob(f"{DRIVE}/05-epi/treino/val/{classe}/*")
              if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
random.shuffle(prova)
prova = prova[:8]

fig, axs = plt.subplots(2, 4, figsize=(22, 11))
acertos = 0
for ax, (arq, gabarito) in zip(axs.ravel(), prova):
    p = especialista.predict(arq, verbose=False)[0]
    palpite = p.names[int(p.probs.top1)]
    certeza = float(p.probs.top1conf)
    ok = palpite == gabarito
    acertos += ok
    ax.imshow(cv2.cvtColor(cv2.imread(arq), cv2.COLOR_BGR2RGB)); ax.axis("off")
    ax.set_title(f"{palpite} {certeza:.0%}", fontsize=24,
                 color="#3fe0a8" if ok else "#ff5c5c")
fig.suptitle(f"Prova final: {acertos} de {len(prova)} corretas", fontsize=36)
plt.tight_layout(); plt.show()

### Passo 5 · onde ele ainda erra

A matriz de confusão é o slide honesto da IA: ela mostra **que tipo** de erro o
modelo comete. Errar dizendo "tem capacete" quando não tem é muito pior do que o
contrário — e nenhum número de acurácia sozinho conta isso.

In [ ]:
from IPython.display import Image, display
import os
for nome in ["confusion_matrix_normalized.png", "confusion_matrix.png", "results.png"]:
    caminho = os.path.join(str(res.save_dir), nome)
    if os.path.exists(caminho):
        print(nome)
        display(Image(filename=caminho, width=1100))

---

## Se a internet cair no evento

Tudo que este notebook produz já está no seu Drive depois do ensaio:
`05-epi/pesos/epi_best.pt`, os gráficos e a matriz. Grave a tela durante o
ensaio e guarde o vídeo em `99-reserva/` — se o Colab não abrir na hora, você
mostra a gravação e **conta a mesma história**, sem pedir desculpa à plateia.